## A basic web research agent

In [ ]:
!pip install 'strands-agents[openai]' ddgs 'strands-agents-tools[local-chromium-browser]' -q

In [ ]:
import logging

# configure log output
logging.getLogger("strands").setLevel(logging.INFO)
logging.getLogger("strands.models").setLevel(logging.DEBUG)
logging.getLogger("strands.tools.executors").setLevel(logging.DEBUG)

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException

from strands import Agent, tool
from strands.models.openai import OpenAIModel

from google.colab import userdata

In [ ]:
base_url = "https://chat-ai.academiccloud.de/v1"

model_id = "qwen3.5-122b-a10b" # try different models

In [ ]:
#@title API Key Configuration { display-mode: "form" }
# choose one option to set your API key

# 1. Set it directly here and hide the cell content once executed to keep your key private
api_key = "123"

# 2. Read it from an ENV
# api_key = os.environ.get("GWDG_API_KEY")

In [ ]:
@tool
def websearch(
    keywords: str, region: str = "us-en", max_results: int | None = None
) -> str:
    """Search the web to get updated information.
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"


In [ ]:
from strands_tools.browser import LocalChromiumBrowser
browser = LocalChromiumBrowser()

In [ ]:
model = OpenAIModel(client_args={"api_key": api_key, "base_url": base_url}, model_id=model_id, params={"temperature": 0.0})

In [ ]:
search_agent = Agent(
    model=model,
    system_prompt="""You are very helpful and accurate Scientific Paper Search assistant.
    Help users find papers based on the science question or resaearch question they are asking.
    Use the websearch tool to find papers. Focus on arxiv. Cite all the referenced papers with URL in the final response.""",
    tools=[websearch, browser.browser]
)

In [ ]:
resp = search_agent("What is the lowest temperature that a sample was imaged at with Low-energy electron microscopy?")

In [ ]:
print(resp.message['content'][0]['text'])